In [18]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from IPython.display import HTML
from matplotlib.ticker import FuncFormatter

# --- Configuration ---
fps = 30
total_duration_sec = 10
pause_duration_sec = 5

# --- Years ---
years = np.array([2023, 2024, 2025], dtype=object)


# --- Data (in %) ---
eigener_bereich = np.array([89.4, 89.3, 92.2])        # Vermögenskunden
gesamtbank      = np.array([93.1, 87.0, 96.5])        # Unternehmenskunden
servicebank     = np.array([87.5, 88.9, 100.0])       # Leasinggesellschaften
kundenbank      = np.array([100.0, 100.0, 100.0])     # Bauträger
gs49            = np.array([np.nan, np.nan, 77.4])    # GS49

# --- Equal spacing for all years ---
x_pos = np.arange(len(years))
x_shifted = x_pos - 0.5

# --- Auto-calc smooth frames ---
total_frames = total_duration_sec * fps
smooth_frames = total_frames // (len(years) - 1)
pause_frames = int(pause_duration_sec * fps)
total_frames_with_pause = total_frames + pause_frames

print(f"🎞️ FPS fixed at {fps}")
print(f"→ smooth_frames per year = {smooth_frames}, total frames = {total_frames}")

# --- Figure setup ---
fig, ax = plt.subplots(figsize=(15, 6))
fig.set_size_inches(13, 5)
ax.set_position([0.05, 0.1, 0.93, 0.83])

ax.set_ylim(75, 104)
ax.set_yticks(np.arange(75, 104, 10))
ax.grid(True, alpha=0.3, color="#404040")
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y)}%"))
fig.patch.set_facecolor("#eaeaef")
ax.set_facecolor("#eaeaef")
ax.margins(x=0, y=0)

ax.set_xlim(-1, len(years) - 1)
ax.set_xticks(x_pos)
ax.grid(True, which="major", axis="x", color="#555557", linewidth=1)
ax.set_xticks(x_shifted, minor=True)
ax.set_xticklabels(years, minor=True)

ax.tick_params(axis="x", which="major", bottom=False, labelbottom=False)
ax.tick_params(axis="x", which="minor", bottom=False, labelsize=10, pad=10, colors="#404040")
ax.tick_params(axis="y", pad=10)

ax.spines["right"].set_visible(True)
ax.spines["top"].set_visible(False)
for spine in ax.spines.values():
    spine.set_color((0.25, 0.25, 0.25, 0.3))
    spine.set_linewidth(1)

for tick in ax.xaxis.get_ticklabels():
    tick.set_color("#555557")
for tick in ax.yaxis.get_ticklabels():
    tick.set_color("#555557")

# --- Lines ---

(line_eb,) = ax.plot([], [], marker="o", linewidth=3, color="#C82E3F",
                     markerfacecolor="white", markeredgewidth=6, markersize=17)

(line_gb,) = ax.plot([], [], marker="o", linewidth=3, color="#96232F",
                     markeredgewidth=6, markersize=17, markerfacecolor="white")

(line_sb,) = ax.plot([], [], marker="o", linewidth=3, color="#FF0000",
                     markerfacecolor="white", markeredgewidth=6, markersize=17)

(line_kb,) = ax.plot([], [], marker="o", linewidth=3, color="#FFC000",
                     markerfacecolor="white", markeredgewidth=6, markersize=17)

(line_gs,) = ax.plot([], [], marker="o", linewidth=3, color="#B7620C",
                     markerfacecolor="white", markeredgewidth=6, markersize=17)

# --- Moving points ---
pt_eb = ax.plot([], [], marker="o", markersize=17, color="#C82E3F",
                markerfacecolor="white", markeredgewidth=6)[0]
pt_gb = ax.plot([], [], marker="o", markersize=17, color="#96232F",
                markerfacecolor="white", markeredgewidth=6)[0]
pt_sb = ax.plot([], [], marker="o", markersize=17, color="#FF0000",
                markerfacecolor="white", markeredgewidth=6)[0]
pt_kb = ax.plot([], [], marker="o", markersize=17, color="#FFC000",
                markerfacecolor="white", markeredgewidth=6)[0]
pt_gs = ax.plot([], [], marker="o", markersize=17, color="#B7620C",
                markerfacecolor="white", markeredgewidth=6)[0]

# --- Init ---
def init():
    for ln in (line_eb, line_gb, line_sb, line_kb, line_gs):
        ln.set_data([], [])
    for pt in (pt_eb, pt_gb, pt_sb, pt_kb, pt_gs):
        pt.set_data([], [])
    return (line_eb, line_gb, line_sb, line_kb, line_gs,
            pt_eb, pt_gb, pt_sb, pt_kb, pt_gs)

# --- Animation ---
def animate(i):

    # Detect pause section → force i to last movement frame
    last_frame = (len(years) - 1) * smooth_frames - 1
    if i >= last_frame:
        i = last_frame

    year_idx = i / smooth_frames
    lower = int(np.floor(year_idx))
    upper = min(lower + 1, len(years) - 1)
    frac = year_idx - lower

    x_now = x_shifted[lower] + frac * (x_shifted[upper] - x_shifted[lower])

    eb_now = eigener_bereich[lower] + frac * (eigener_bereich[upper] - eigener_bereich[lower])
    gb_now = gesamtbank[lower]      + frac * (gesamtbank[upper] - gesamtbank[lower])
    sb_now = servicebank[lower]     + frac * (servicebank[upper] - servicebank[lower])
    kb_now = kundenbank[lower]      + frac * (kundenbank[upper] - kundenbank[lower])
    # --- GS49: show only at last year ---
    # --- GS49: show only when 2025 is reached ---
    # --- GS49: show only at final year (2025) ---
    if i >= last_frame:
        line_gs.set_data([x_shifted[-1]], [gs49[-1]])
        pt_gs.set_data([x_shifted[-1]], [gs49[-1]])
    else:
        line_gs.set_data([], [])
        pt_gs.set_data([], [])




    x_full = x_shifted[:upper + 1]

    line_eb.set_data(np.append(x_full[:-1], x_now),
                     np.append(eigener_bereich[:upper], eb_now))

    line_gb.set_data(np.append(x_full[:-1], x_now),
                     np.append(gesamtbank[:upper], gb_now))

    line_sb.set_data(np.append(x_full[:-1], x_now),
                     np.append(servicebank[:upper], sb_now))

    line_kb.set_data(np.append(x_full[:-1], x_now),
                     np.append(kundenbank[:upper], kb_now))

    pt_eb.set_data([x_now], [eb_now])
    pt_gb.set_data([x_now], [gb_now])
    pt_sb.set_data([x_now], [sb_now])
    pt_kb.set_data([x_now], [kb_now])

# --- Build animation ---
interval_ms = 1000 / fps
anim = FuncAnimation(fig, animate, init_func=init,
                     frames=total_frames_with_pause, interval=interval_ms,
                     blit=False, repeat=True)
# --- Save MP4 ---
mp4_path = "1.mp4"
writer_mp4 = FFMpegWriter(fps=fps, codec="libx264", bitrate=2500, extra_args=["-pix_fmt", "yuv420p"])
anim.save(mp4_path, writer=writer_mp4, dpi=300)
print(f"✅ Saved MP4: {mp4_path}")

plt.close(fig)
plt.draw()
plt.close(fig)

HTML(anim.to_html5_video())


🎞️ FPS fixed at 30
→ smooth_frames per year = 150, total frames = 300
✅ Saved MP4: 1.mp4


<Figure size 640x480 with 0 Axes>

In [19]:
from google.colab import files
files.download('1.mp4')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>